# mainframe-rag on Kaggle — Path A (GPU, full answers + eval numbers)

Deploys the complete application stack natively in one Kaggle notebook: Qdrant 1.19.0 binary, vLLM embedding server, vLLM reasoning server, FastAPI agent — then ingests **your** PDF dataset, runs interactive Q&A, and records retrieval + answer-tier eval numbers.

## Before you run anything

1. Notebook Settings: **Accelerator = GPU T4**, **Internet = ON** (pip, Qdrant binary, BM25 weights, Hugging Face models all need it).
2. Attach your PDF dataset to this notebook (Add data → your dataset). Then set `DATASET_DIR` in cell 2 to its `/kaggle/input/...` path.
3. **Legal hard rule (repo policy): only PDFs you own the rights to.** Never ingest vendor manuals (IBM, Broadcom, BMC, Precisely) from a public runner — ingest of licensed manuals happens inside the enterprise air-gap only.
4. Qwen models are access-gated on Hugging Face: accept the license on the model pages, then store a token in **Kaggle → Add-ons → Secrets** as `HF_TOKEN` and enable it for this notebook (cell 3 reads it).
5. Sessions are ephemeral: everything writable lives under `/kaggle/working` (persists as notebook output). GPU quota is weekly — run cells top-to-bottom, and run the last cell to stop servers when done.

What this does **not** do: Docker, Kind, Helm, air-gap packaging (`make airgap-*`, `make sim`) — none of those exist on Kaggle. Ports are not publicly reachable; you talk to the agent from notebook cells.

In [ ]:
# Cell 2 — run configuration. EDIT DATASET_DIR, then Run All.
import os
from pathlib import Path

WORK = Path("/kaggle/working/rag")          # all writable state lives here
REPO = WORK / "qdrant-pdf-rag"               # repo checkout
REPO_SHA = "bd8186b4652cfdb24736bee92a910f20235839e1"  # pinned green main; unpin to track main
QDRANT_VERSION = "1.19.0"                    # must match images.txt pin
VLLM_VERSION = "0.28.0"                      # must match scripts/run_local_vllm.sh IMAGE pin
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
DENSE_DIM = "1024"
REASON_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
USER_COLLECTION = "user-manuals"             # your PDFs live here (agent default stays on the eval corpus)
DATASET_DIR = "/kaggle/input/<your-dataset-slug>"  # <-- EDIT ME

WORK.mkdir(parents=True, exist_ok=True)
os.environ["DATASET_DIR"] = DATASET_DIR  # %%bash cells inherit os.environ
pdfs = [p for p in Path(DATASET_DIR).rglob("*") if p.suffix.lower() == ".pdf"] if Path(DATASET_DIR).exists() else []
assert pdfs, f"No PDFs under {DATASET_DIR}: attach your dataset and set DATASET_DIR"
print(f"dataset PDFs: {len(pdfs)}  work: {WORK}")

In [ ]:
# Cell 3 — preflight: GPU present, HF_TOKEN available for the gated Qwen weights.
import shutil, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as exc:
    raise RuntimeError("Add an HF_TOKEN Kaggle secret (Add-ons -> Secrets) with access to the Qwen models") from exc

In [ ]:
# Cell 4 — Python 3.14 (repo requires >=3.14) + locked dependencies.
import subprocess
uv = Path.home() / ".local/bin/uv"
if not uv.exists():
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
subprocess.run([str(uv), "python", "install", "3.14"], check=True)
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/yonatan895/qdrant-pdf-rag", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", REPO_SHA], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", REPO_SHA], check=True)
VENV_PY = REPO / ".venv/bin/python"
if not VENV_PY.exists():
    subprocess.run([str(uv), "venv", "--python", "3.14", str(REPO / ".venv")], check=True)
    subprocess.run([str(VENV_PY), "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([str(VENV_PY), "-m", "pip", "install", "-r", str(REPO / "requirements.lock.txt"), "-e", f"{REPO}[dev]"], check=True)
print(subprocess.run([str(VENV_PY), "-c", "import sys, mainframe_rag; print(sys.version.split()[0], 'repo OK')"], capture_output=True, text=True, cwd=REPO).stdout)

In [ ]:
# Cell 5 — Qdrant server binary (no Docker on Kaggle). Resolves the release
# asset via the GitHub API so the cell survives asset renames; fail-closed on version.
import json, subprocess, tarfile, urllib.request

BIN_DIR = WORK / "qdrant-bin"
STORE = WORK / "qdrant-storage"
BIN_DIR.mkdir(exist_ok=True); STORE.mkdir(exist_ok=True)
QBIN = BIN_DIR / "qdrant"
if not QBIN.exists():
    with urllib.request.urlopen(f"https://api.github.com/repos/qdrant/qdrant/releases/tags/v{QDRANT_VERSION}", timeout=60) as r:
        assets = json.load(r)["assets"]
    cands = [a["browser_download_url"] for a in assets if "linux" in a["name"] and a["name"].endswith(".tar.gz")]
    assert cands, "no linux tarball asset found"
    url = next((u for u in cands if "musl" in u), cands[0])
    print("downloading", url)
    urllib.request.urlretrieve(url, WORK / "qdrant.tgz")
    with tarfile.open(WORK / "qdrant.tgz") as t:
        t.extractall(BIN_DIR, filter="data")
    assert QBIN.exists(), "qdrant binary missing after extract"

LOG = open(WORK / "qdrant.log", "ab")
QP = subprocess.Popen([str(QBIN), "--storage-path", str(STORE)], stdout=LOG, stderr=subprocess.STDOUT, start_new_session=True)
print("qdrant pid", QP.pid)

import time
t0 = time.time()
while time.time() - t0 < 180:
    try:
        with urllib.request.urlopen("http://127.0.0.1:6333/readyz", timeout=5) as r:
            if r.status == 200:
                print("qdrant ready"); break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError("qdrant not ready; see /kaggle/working/rag/qdrant.log")

In [ ]:
%%bash
# Cell 6 — BM25 sparse weights (verified against the in-repo sha256 manifest).
cd /kaggle/working/rag/qdrant-pdf-rag && make bm25-weights

In [ ]:
# Cell 7 — vLLM (pinned to the repo's local-GPU image tag). Multi-GB download, one time.
import subprocess
VENV_PY = REPO / ".venv/bin/python"
subprocess.run([str(VENV_PY), "-m", "pip", "install", f"vllm=={VLLM_VERSION}"], check=True)
print("vllm installed")

In [ ]:
# Cell 8 — embedding server :8001 (pooling runner, 4096 window per repo runbook).
import subprocess, time, urllib.request
VENV_PY = REPO / ".venv/bin/python"
LOG = open(WORK / "vllm-embed.log", "ab")
EP = subprocess.Popen([str(VENV_PY), "-m", "vllm.entrypoints.openai.api_server",
    "--model", EMBED_MODEL, "--runner", "pooling", "--convert", "embed",
    "--enforce-eager", "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.45", "--port", "8001"],
    stdout=LOG, stderr=subprocess.STDOUT, start_new_session=True, cwd=REPO)
print("embed pid", EP.pid)
t0 = time.time()
while time.time() - t0 < 1200:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8001/v1/models", timeout=10) as r:
            if r.status == 200:
                print("embed server ready"); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("embed server not ready; see vllm-embed.log (OOM? lower --gpu-memory-utilization)")

## Your dataset → Qdrant

Your PDFs land in their own `user-manuals` collection so the eval corpus (default `mainframe_manuals` collection) stays pristine for the numbers below. Each `%%bash` cell below is a fresh shell, so `VAR=...` prefixes are scoped to that one command — never a global export.

In [ ]:
%%bash
# Cell 10 — ingest smoke: 5 docs first (--limit is a stable prefix, so the full run resumes).
cd /kaggle/working/rag/qdrant-pdf-rag && \
QDRANT_URL=http://127.0.0.1:6333 QDRANT_COLLECTION=user-manuals \
EMBED_MODE=vllm EMBED_BASE_URL=http://127.0.0.1:8001/v1 EMBED_MODEL=Qwen/Qwen3-Embedding-0.6B DENSE_DIM=1024 \
PYTHONPATH=src .venv/bin/python -m mainframe_rag.ingest.run_ingest \
  --src "$DATASET_DIR" --progress /kaggle/working/rag/user-progress.jsonl --workers 2 --limit 5

In [ ]:
%%bash
# Cell 11 — full ingest of your dataset (inventory skips the 5 already done).
cd /kaggle/working/rag/qdrant-pdf-rag && \
QDRANT_URL=http://127.0.0.1:6333 QDRANT_COLLECTION=user-manuals \
EMBED_MODE=vllm EMBED_BASE_URL=http://127.0.0.1:8001/v1 EMBED_MODEL=Qwen/Qwen3-Embedding-0.6B DENSE_DIM=1024 \
PYTHONPATH=src .venv/bin/python -m mainframe_rag.ingest.run_ingest \
  --src "$DATASET_DIR" --progress /kaggle/working/rag/user-progress.jsonl --workers 2 && \
curl -s http://127.0.0.1:6333/collections/user-manuals | .venv/bin/python -c "import json,sys; print('points:', json.load(sys.stdin)['result']['points_count'])"

In [ ]:
# Cell 12 — synthetic eval corpus: generated at runtime from the dev golden set
# (zero committed PDFs), ingested into the DEFAULT collection for the gates below.
import json, subprocess, sys
sys.path.insert(0, str(REPO / "scripts"))
from pathlib import Path
from gate_l1 import generate_synthetic_golden_corpus
entries = [json.loads(l) for l in open(REPO / "evals/golden.jsonl") if l.strip()]
corpus = WORK / "synth-corpus"
generate_synthetic_golden_corpus(entries, corpus)
env = dict(QDRANT_URL="http://127.0.0.1:6333", EMBED_MODE="vllm",
           EMBED_BASE_URL="http://127.0.0.1:8001/v1", EMBED_MODEL=EMBED_MODEL, DENSE_DIM=DENSE_DIM)
import os
subprocess.run([str(REPO / ".venv/bin/python"), "-m", "mainframe_rag.ingest.run_ingest",
    "--src", str(corpus), "--progress", str(WORK / "synth-progress.jsonl"), "--workers", "2"],
    check=True, cwd=REPO, env={**os.environ, **env, "PYTHONPATH": "src"})
print("synthetic corpus ingested into default collection")

In [ ]:
# Cell 13 — reasoning server :8000, then the agent :8080 on the DEFAULT (eval) collection.
import subprocess, time, urllib.request
VENV_PY = REPO / ".venv/bin/python"
LOG = open(WORK / "vllm-reason.log", "ab")
RP = subprocess.Popen([str(VENV_PY), "-m", "vllm.entrypoints.openai.api_server",
    "--model", REASON_MODEL, "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.45", "--port", "8000"],
    stdout=LOG, stderr=subprocess.STDOUT, start_new_session=True, cwd=REPO)
print("reason pid", RP.pid)
t0 = time.time()
while time.time() - t0 < 1200:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=10) as r:
            if r.status == 200:
                print("reasoning server ready"); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("reasoning server not ready; see vllm-reason.log")

ALOG = open(WORK / "agent.log", "ab")
import os
agent_env = {**os.environ, "LLM_STREAM": "true", "QDRANT_URL": "http://127.0.0.1:6333",
    "EMBED_MODE": "vllm", "EMBED_BASE_URL": "http://127.0.0.1:8001/v1",
    "EMBED_MODEL": EMBED_MODEL, "DENSE_DIM": DENSE_DIM,
    "LLM_BASE_URL": "http://127.0.0.1:8000/v1", "LLM_MODEL_REASONING": REASON_MODEL}
AP = subprocess.Popen([str(VENV_PY), "-m", "uvicorn", "mainframe_rag.agent.app:app",
    "--host", "127.0.0.1", "--port", "8080"], stdout=ALOG, stderr=subprocess.STDOUT,
    start_new_session=True, cwd=REPO, env=agent_env)
print("agent pid", AP.pid)
t0 = time.time()
while time.time() - t0 < 300:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/healthz", timeout=10) as r:
            body = r.read().decode()
            if r.status == 200:
                print("agent", body); break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("agent not ready; see agent.log")

In [ ]:
# Cell 14 — smoke: /v1/search needs no LLM; /v1/answer uses the reasoning model.
import json, urllib.request

def post(path, payload):
    req = urllib.request.Request("http://127.0.0.1:8080" + path,
        data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

s = post("/v1/search", {"query": "What does message IEF450I mean?", "limit": 3})
print("search kind:", s["query_kind"])
for h in s["hits"]:
    print(f"  [{h['score']:.3f}] {h['doc_id']} :: {' > '.join(h['heading_path'][:2])} (p. {h['page_label']})")
a = post("/v1/answer", {"query": "What does message IEF450I mean?"})
print("\nanswer:", a["answer"][:500])
print("citations:", a["citations"])

## Eval numbers

Retrieval accuracy vs the committed vLLM baseline, then answer-tier grounding on a bounded sample. The frozen holdout (`evals/holdout.jsonl`) is **excluded on purpose** — release candidates only, never iterated from a demo box.

In [ ]:
%%bash
# Cell 16 — retrieval: recall@k / MRR vs evals/baseline-vllm.json (mode-keyed by EMBED_MODE=vllm).
cd /kaggle/working/rag/qdrant-pdf-rag && \
QDRANT_URL=http://127.0.0.1:6333 EMBED_MODE=vllm \
EMBED_BASE_URL=http://127.0.0.1:8001/v1 EMBED_MODEL=Qwen/Qwen3-Embedding-0.6B DENSE_DIM=1024 \
make eval && cat bundles/eval-summary.md

In [ ]:
%%bash
# Cell 17 — answer tier: grounding honesty on a 12-query stratified sample
# (one reasoning call per query; raise N for tighter numbers, ~1-2 min/query on T4).
# --golden pins the DEV set only: the frozen holdout is never scored here.
cd /kaggle/working/rag/qdrant-pdf-rag && \
QDRANT_URL=http://127.0.0.1:6333 EMBED_MODE=vllm \
EMBED_BASE_URL=http://127.0.0.1:8001/v1 EMBED_MODEL=Qwen/Qwen3-Embedding-0.6B DENSE_DIM=1024 \
LLM_BASE_URL=http://127.0.0.1:8000/v1 LLM_MODEL_REASONING=Qwen/Qwen2.5-0.5B-Instruct \
.venv/bin/python scripts/eval_answers.py --golden evals/golden.jsonl --max-queries 12 \
  --out /kaggle/working/rag/eval-answers-report.json --summary /kaggle/working/rag/eval-answers-summary.md && \
cat /kaggle/working/rag/eval-answers-summary.md

## Interactive Q&A on your dataset

`query_demo.py --collection user-manuals` reads your collection while the agent keeps serving the eval corpus. Edit `Q` and run: `--answer` for a grounded reasoning answer, drop it for search hits.

In [ ]:
%%bash
# Cell 19 — ask your manuals. Edit Q, re-run. Remove --answer for search-only hits.
Q="How do I resolve a dataset contention abend?"
cd /kaggle/working/rag/qdrant-pdf-rag && \
PYTHONPATH=src .venv/bin/python scripts/query_demo.py --answer --query "$Q" --collection user-manuals --limit 3 \
  --embed-mode vllm --embed-url http://127.0.0.1:8001/v1 --embed-model Qwen/Qwen3-Embedding-0.6B --dense-dim 1024 \
  --vllm-url http://127.0.0.1:8000/v1 --model Qwen/Qwen2.5-0.5B-Instruct

In [ ]:
# Cell 20 — keep the evidence, stop the servers (frees GPU quota).
import subprocess, urllib.request
for coll in ("user-manuals", "mainframe_manuals"):
    req = urllib.request.Request(f"http://127.0.0.1:6333/collections/{coll}/snapshots", method="POST")
    try:
        print(coll, urllib.request.urlopen(req, timeout=120).read().decode()[:120])
    except Exception as exc:
        print(coll, "snapshot skipped:", exc)
print("keep from /kaggle/working/rag: qdrant-storage/, *-progress.jsonl, eval-answers-*, bundles/, *.log")
subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server'; pkill -f 'uvicorn mainframe_rag.agent.app'; pkill -x qdrant", shell=True)
print("servers stopped")